# Hybrid Experiments: Latest XGB + Fusion

Plan... & Notes

### Imports

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
import xgboost as xgb
import numpy as np
import librosa as lb
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold

## Branch 1: **Engineered XGBoost Features**

Dataset now features:

- 12 kHz target sample rate,
- 6-second chunks,
- overlap for non-COPD classes using a 2-second step,
- no overlap for COPD,
- Mu-law compression,
- and a richer feature set including stds, bandwidth, skewness/kurtosis, and MFCCs up to 15.

### Load XGB data

In [2]:
xgb_df = pd.read_csv("/Users/keira/code/mi-mi-mia/smart-stethoscope/raw_data/Extracted features/xgboost_features_6s_12kHz_compressed_after_normalization_w_overlapping_mfcc15.csv")

xgb_df.head()

,rms_mean,rms_std,zcr_mean,centroid_mean,centroid_std,flatness_mean,flatness_std,rolloff_mean,flux_mean,bandwidth_mean,...,mfcc_13_mean,mfcc_13_std,mfcc_14_mean,mfcc_14_std,mfcc_15_mean,mfcc_15_std,patient_id,chunk_id,original_file,diagnosis
0,0.740132,0.083106,0.001316,98.066230,93.868556,0.000034,0.000146,119.431516,1.826715,340.148743,...,8.284259,2.947182,8.153688,3.566122,7.156815,3.025179,223,0,223_1b1_Pr_sc_Meditron.wav,COPD
1,0.696974,0.079933,0.001572,94.479942,81.417895,0.000041,0.000361,107.338763,1.494651,338.655438,...,8.452517,3.363525,7.557367,4.090311,6.353885,4.028513,223,1,223_1b1_Pr_sc_Meditron.wav,COPD
2,0.670958,0.079359,0.001590,90.514388,75.801911,0.000024,0.000205,99.817154,1.613105,339.537651,...,8.950355,2.893862,7.780566,3.040372,6.503709,3.147859,223,2,223_1b1_Pr_sc_Meditron.wav,COPD
3,0.675636,0.093185,0.001153,81.297469,52.764923,0.000008,0.000023,81.740359,1.693879,322.028853,...,8.722843,2.706289,8.697692,3.737317,6.902027,3.432181,223,3,223_1b1_Pr_sc_Meditron.wav,COPD
4,0.635533,0.069224,0.001202,80.768717,67.285689,0.000014,0.000083,81.865027,1.705645,325.092538,...,9.458111,3.485257,10.628664,4.948114,8.951859,4.647463,223,4,223_1b1_Pr_sc_Meditron.wav,COPD


In [3]:
xgb_df.shape, xgb_df.columns, xgb_df.dtypes

((3606, 46),
 Index(['rms_mean', 'rms_std', 'zcr_mean', 'centroid_mean', 'centroid_std',
        'flatness_mean', 'flatness_std', 'rolloff_mean', 'flux_mean',
        'bandwidth_mean', 'skewness_mean', 'kurtosis_mean', 'mfcc_1_mean',
        'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std',
        'mfcc_4_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_5_std', 'mfcc_6_mean',
        'mfcc_6_std', 'mfcc_7_mean', 'mfcc_7_std', 'mfcc_8_mean', 'mfcc_8_std',
        'mfcc_9_mean', 'mfcc_9_std', 'mfcc_10_mean', 'mfcc_10_std',
        'mfcc_11_mean', 'mfcc_11_std', 'mfcc_12_mean', 'mfcc_12_std',
        'mfcc_13_mean', 'mfcc_13_std', 'mfcc_14_mean', 'mfcc_14_std',
        'mfcc_15_mean', 'mfcc_15_std', 'patient_id', 'chunk_id',
        'original_file', 'diagnosis'],
       dtype='object'),
 rms_mean          float64
 rms_std           float64
 zcr_mean          float64
 centroid_mean     float64
 centroid_std      float64
 flatness_mean     float64
 flatness_std      float64


### XGB preprocessing

#### Apply 5-class filter

In [6]:
# Filter to same 5 classes - COPD, pneumonia, healthy, URTI, bronchiectasis
classes_to_keep = ["COPD", "Pneumonia", "Healthy", "URTI", "Bronchiectasis"]

xgb_df_filtered = xgb_df[xgb_df["diagnosis"].isin(classes_to_keep)].copy()
xgb_df_filtered = xgb_df_filtered.reset_index(drop=True)

xgb_df_filtered["diagnosis"].value_counts()

#### Label-encode target

In [10]:
# Encode target
le = LabelEncoder()
xgb_df_filtered["target"] = le.fit_transform(xgb_df_filtered["diagnosis"])

# Create a dictionary mapping labels -> numbers
class_mapping = dict(zip(le.classes_, le.transform(le.classes_)))

class_mapping

{'Bronchiectasis': 0, 'COPD': 1, 'Healthy': 2, 'Pneumonia': 3, 'URTI': 4}

### Train & Test with Grouped Cross Validation

In [ ]:
# Define features and target
X = xgb_df_filtered.drop(columns=["original_file", "patient_id", "diagnosis", "target", "chunk_id"])
y = xgb_df_filtered["target"]

# Define groups (by patient)
groups = xgb_df_filtered["patient_id"]

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Number of groups (patients): {groups.nunique()}")
print("\nConfirm feature columns:")
print(list(X.columns))

In [ ]:
# Grouped Cross Validation (split by patient ID)
# IMPORTANT: We used StratifiedGroupKFold to preserve patient-level separation
# while improving class balance across folds.
sgkf = StratifiedGroupKFold(n_splits=3)

fold_results = []

for fold, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups=groups), start=1):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}")
    print(f"{'='*60}")

    # Outer split: creates grouped train/test splits
    X_train_full = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train_full = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    groups_train_full = groups.iloc[train_idx]

    # Inner split: creates grouped train/validation splits
    gss_val = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
    idx_train, idx_val = next(gss_val.split(X_train_full, y_train_full, groups=groups_train_full))

    X_train = X_train_full.iloc[idx_train]
    X_val = X_train_full.iloc[idx_val]

    y_train = y_train_full.iloc[idx_train]
    y_val = y_train_full.iloc[idx_val]

    # Class-balance sample weights on TRAIN only
    w_train = compute_sample_weight(class_weight="balanced", y=y_train)

    # ==================
    # XGBoost model
    # ==================
    # UPDATED to match latest model from Antonella
    xgb_model = xgb.XGBClassifier(
        n_estimators=600,
        max_depth=3,
        subsample=0.8,        # only 80% of lines in each tree
        colsample_bytree=0.7, # only 70% of cols in each tree
        reg_lambda=1.5,
        objective='multi:softprob',
        num_class=len(le.classes_),
        random_state=42,
        eval_metric='mlogloss',
        early_stopping_rounds=15
    )

    print("Train classes:", np.sort(y_train.unique()))
    print("Val classes:  ", np.sort(y_val.unique()))
    print("Test classes: ", np.sort(y_test.unique()))

    # Train
    xgb_model.fit(
        X_train,
        y_train,
        sample_weight=w_train, # gives more importance to rare classes
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    # Predict
    y_pred = xgb_model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    weighted_f1 = f1_score(y_test, y_pred, average="weighted")

    print(f"Fold {fold} accuracy   : {acc:.4f}")
    print(f"Fold {fold} macro F1   : {macro_f1:.4f}")
    print(f"Fold {fold} weighted F1: {weighted_f1:.4f}\n")

    print(classification_report(
        y_test,
        y_pred,
        target_names=le.classes_,
        zero_division=0
    ))

    fold_results.append({
        "fold": fold,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    })

# =========================================================
# Summary across folds
# =========================================================
results_df = pd.DataFrame(fold_results)

print("\n" + "="*60)
print("CROSS-VALIDATION SUMMARY")
print("="*60)
print(results_df)

print("\nMean metrics:")
print(results_df[["accuracy", "macro_f1", "weighted_f1"]].mean())



Feature matrix shape: (3478, 42)
Target shape: (3478,)
Number of groups (patients): 117

Confirm feature columns:
['rms_mean', 'rms_std', 'zcr_mean', 'centroid_mean', 'centroid_std', 'flatness_mean', 'flatness_std', 'rolloff_mean', 'flux_mean', 'bandwidth_mean', 'skewness_mean', 'kurtosis_mean', 'mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_5_std', 'mfcc_6_mean', 'mfcc_6_std', 'mfcc_7_mean', 'mfcc_7_std', 'mfcc_8_mean', 'mfcc_8_std', 'mfcc_9_mean', 'mfcc_9_std', 'mfcc_10_mean', 'mfcc_10_std', 'mfcc_11_mean', 'mfcc_11_std', 'mfcc_12_mean', 'mfcc_12_std', 'mfcc_13_mean', 'mfcc_13_std', 'mfcc_14_mean', 'mfcc_14_std', 'mfcc_15_mean', 'mfcc_15_std']

FOLD 1
Train classes: [0 1 2 3 4]
Val classes:   [1 2 3 4]
Test classes:  [0 1 2 3 4]
Fold 1 accuracy   : 0.7185
Fold 1 macro F1   : 0.4369
Fold 1 weighted F1: 0.7377

                precision    recall  f1-score   support

Bronchiectasis       0.58    

## Branch 2: **CNN**

1. Quick baseline using yesterday's CNN model (trained to match previous XGBoost)
2. Adapt to latest preprocessing in XGBoost